# RL Test Flow Optimization — Notebook 3 of 3: Final + Plots + Export

**Prerequisites**: `rl-stage1-results` AND `rl-stage2-results` datasets added as inputs.
**RL training device**: CPU (intentional for SB3 MLP workloads).

| Step | Description | Est. Time |
|------|-------------|----------|
| 8 | Final training: best algo + HPO params × 1M steps | ~3-4 h |
| 9 | Curriculum: n_tests=[10,50,200] × 200K each | ~1-2 h |
| 10 | Learning curves + reward distribution | ~5 min |
| 11 | Coverage histogram + cost efficiency | ~5 min |
| 12 | Heatmap + comprehensive summary table | ~5 min |
| 13 | ZIP all models + export | ~5 min |


## Step 0: Reinstall + Load All Results

In [ ]:
import subprocess, sys, os, json
subprocess.run(['pip', 'install', '-q',
    'stable-baselines3[extra]', 'sb3-contrib', 'gymnasium',
    'optuna', 'mlflow', 'pyarrow', 'matplotlib', 'seaborn', 'numpy', 'torch'], check=True)

!rm -rf rl-test-flow-optimization
!git clone https://github.com/rajendarmuddasani/rl-test-flow-optimization.git
os.chdir('rl-test-flow-optimization')
sys.path.insert(0, '.')

import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — not required for this run"}')
print('RL train device: cpu (forced intentionally for SB3 MlpPolicy)')

NB2_PATH = '/kaggle/input/rl-stage2-results/stage2_results.json'
with open(NB2_PATH) as f:
    nb2 = json.load(f)

baseline_results  = nb2['baselines']
stage1_results    = nb2['stage1']
stage2_results    = nb2['stage2']
hpo_result        = nb2['hpo']
top2_algos        = nb2['top2_algos']
best_algo         = nb2['best_algo']
best_bl           = nb2['best_base_reward']
best_params       = nb2['best_params']
baseline_df       = pd.DataFrame(baseline_results).T
stage1_df         = pd.DataFrame(stage1_results).T

print(f'All results loaded. Best algo: {best_algo} with params:')
for k, v in best_params.items(): print(f'  {k}: {v}')


## Step 8: Final Training — 1M Steps with Best Config

1 million timesteps is the production-grade benchmark for this environment size.
Using the best algorithm and HPO-tuned hyperparameters from NB2.

In [ ]:
import time as _time
from src.environment import TestFlowEnv
from src.agent import ALGO_REGISTRY, evaluate_trained_model
from pathlib import Path

FINAL_STEPS = 1_000_000
SAVE_DIR3 = Path('/kaggle/working/rl_stage3')
SAVE_DIR3.mkdir(parents=True, exist_ok=True)

print(f'Final Training: {FINAL_STEPS:,} steps | algo={best_algo}')
print(f'Params: {best_params}')
print('=' * 70)

env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
train_fn = ALGO_REGISTRY[best_algo]
t0 = _time.time()
final_model = train_fn(
    env,
    total_timesteps=FINAL_STEPS,
    output_dir=str(SAVE_DIR3 / 'final_model'),
    seed=42,
    **{k: v for k, v in best_params.items() if k not in ['net_arch']},
)
final_train_time = _time.time() - t0

final_metrics = evaluate_trained_model(env, final_model, n_episodes=200)
print(f'\nFinal Model (1M steps, 1000 eval episodes):')
print(f'  Mean Reward:    {final_metrics["mean_reward"]:+.2f}')
print(f'  Std Reward:     {final_metrics["std_reward"]:+.2f}')
print(f'  Accuracy:       {final_metrics["accuracy"]*100:.1f}%')
print(f'  Defect Rate:    {final_metrics.get("defect_escape_rate", 0)*100:.2f}%')
print(f'  Train Time:     {final_train_time/3600:.2f}h')
print(f'  vs Baseline:   +{final_metrics["mean_reward"] - best_bl:.2f} improvement')


## Step 9: Curriculum Learning — Complexity Ramp [10 → 50 → 200 tests]

Progressive complexity training: start with easy (10 tests), grow to full scale (200).
This is the same principle used in AlphaGo, GPT curriculum, and AMD test chip triage.

In [ ]:
curriculum_results = {}
CURR_STEPS = 200_000  # per stage
STAGES = [10, 50, 200]

print(f'Curriculum: stages={STAGES}, steps_per_stage={CURR_STEPS:,}')
print('=' * 70)

for n_tests in STAGES:
    print(f'\n>>> Stage n_tests={n_tests}')
    env = TestFlowEnv(n_tests=n_tests, cost_budget=50.0, time_budget=120.0)
    t0 = _time.time()
    model = train_fn(
        env,
        total_timesteps=CURR_STEPS,
        output_dir=str(SAVE_DIR3 / f'curriculum_{n_tests}'),
        seed=42,
    )
    elapsed = _time.time() - t0
    metrics = evaluate_trained_model(env, model, n_episodes=100)
    curriculum_results[n_tests] = {**metrics, 'train_time_sec': round(elapsed, 1)}
    print(f'  reward={metrics["mean_reward"]:+.2f} | acc={metrics["accuracy"]*100:.1f}% | {elapsed/60:.1f}m')

print('\nCurriculum complete:', curriculum_results.keys())


## Step 10: Learning Curves + Reward Distribution

In [ ]:
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

# Reconstruct stage2 seed data for plotting
fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel A: Stage-1 comparison bars
ax0 = fig.add_subplot(gs[0, 0])
algos_s1 = list(stage1_results.keys())
means_s1 = [stage1_results[a]['mean_reward'] for a in algos_s1]
accs_s1  = [stage1_results[a]['accuracy']    for a in algos_s1]
x = np.arange(len(algos_s1))
bars = ax0.bar(x, means_s1, color=['#2ECC71','#3498DB','#E74C3C','#9B59B6'])
ax0.axhline(best_bl, color='black', linestyle='--', linewidth=2, label=f'Baseline ({best_bl:.1f})')
ax0.set_xticks(x); ax0.set_xticklabels([a.upper() for a in algos_s1], rotation=45, ha='right')
ax0.set_title('Stage-1 Rewards (200K steps)', fontsize=12, fontweight='bold')
ax0.set_ylabel('Mean Reward'); ax0.legend()
for bar, m in zip(bars, means_s1): ax0.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{m:.1f}', ha='center', fontsize=8)

# Panel B: Stage-2 seed stability
ax1 = fig.add_subplot(gs[0, 1])
colors = ['#2ECC71', '#E74C3C']
for i, algo in enumerate(top2_algos):
    rewards = [m['mean_reward'] for m in stage2_results[algo]]
    ax1.boxplot(rewards, positions=[i+1], widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=colors[i], alpha=0.7))
ax1.set_xticks([1, 2]); ax1.set_xticklabels([a.upper() for a in top2_algos], rotation=45)
ax1.axhline(best_bl, color='black', linestyle='--', linewidth=2)
ax1.set_title('Stage-2 Seed Stability (500K × 3 seeds)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Reward per Seed')

# Panel C: Curriculum progression
ax2 = fig.add_subplot(gs[0, 2])
curr_n = list(curriculum_results.keys())
curr_r = [curriculum_results[n]['mean_reward'] for n in curr_n]
curr_a = [curriculum_results[n]['accuracy']    for n in curr_n]
ax2.plot(curr_n, curr_r, 'o-', color='#3498DB', linewidth=2.5, markersize=8, label='Reward')
ax2b = ax2.twinx()
ax2b.plot(curr_n, curr_a, 's--', color='#E74C3C', linewidth=2, markersize=8, label='Accuracy')
ax2.set_xlabel('Number of Tests (Curriculum Stage)'); ax2.set_ylabel('Mean Reward', color='#3498DB')
ax2b.set_ylabel('Accuracy', color='#E74C3C')
ax2.set_title('Curriculum Learning Progression', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left'); ax2b.legend(loc='upper right')

# Panel D: HPO optimization history
ax3 = fig.add_subplot(gs[1, 0])
if 'trial_values' in hpo_result:
    vals = hpo_result['trial_values']
    best_so_far = [max(vals[:i+1]) for i in range(len(vals))]
    ax3.plot(vals, alpha=0.5, color='gray', linewidth=1, label='Trial reward')
    ax3.plot(best_so_far, color='#E74C3C', linewidth=2.5, label='Best so far')
    ax3.set_xlabel('Trial #'); ax3.set_ylabel('Reward')
    ax3.set_title('Optuna HPO Convergence', fontsize=12, fontweight='bold')
    ax3.legend()
else:
    ax3.text(0.5, 0.5, 'HPO trial history\nnot serialized', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Optuna HPO Convergence', fontsize=12, fontweight='bold')

# Panel E: All stages comparison radar-bar
ax4 = fig.add_subplot(gs[1, 1])
categories = ['Baseline', 'Stage-1\nbest', 'Stage-2\nbest', 'Final\n1M', 'Curriculum\n200t']
s2_best = max(max(m['mean_reward'] for m in v) for v in stage2_results.values())
values  = [
    best_bl,
    stage1_df['mean_reward'].max() if hasattr(stage1_df, 'columns') else max(stage1_results[a]['mean_reward'] for a in stage1_results),
    s2_best,
    final_metrics['mean_reward'],
    curriculum_results[200]['mean_reward'],
]
clrs = ['#7F8C8D','#5DADE2','#2ECC71','#F39C12','#9B59B6']
ax4.bar(categories, values, color=clrs, edgecolor='white', linewidth=1.5)
ax4.set_title('Reward Progression by Stage', fontsize=12, fontweight='bold')
ax4.set_ylabel('Mean Reward')
for i, v in enumerate(values): ax4.text(i, v + 0.2, f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

# Panel F: Cost efficiency vs reward scatter
ax5 = fig.add_subplot(gs[1, 2])
all_metrics = {
    **{k: v for k, v in baseline_results.items()},
    **{k+'_s1': v for k, v in stage1_results.items()},
    'Final_1M': final_metrics,
}
if all(hasattr(v, 'get') and 'mean_cost_per_ep' in v for v in all_metrics.values()):
    costs = [v.get('mean_cost_per_ep', 0) for v in all_metrics.values()]
    rews  = [v.get('mean_reward', 0) for v in all_metrics.values()]
    ax5.scatter(costs, rews, alpha=0.8, s=100)
    for label, c, r in zip(all_metrics.keys(), costs, rews):
        ax5.annotate(label, (c, r), fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax5.set_xlabel('Mean Cost/Episode'); ax5.set_ylabel('Mean Reward')
else:
    ax5.text(0.5, 0.5, 'Cost-efficiency data\nnot available in metrics', ha='center', va='center', transform=ax5.transAxes)
ax5.set_title('Cost Efficiency vs Reward', fontsize=12, fontweight='bold')

fig.suptitle('RL Test Flow Optimization — Complete Results Summary', fontsize=15, fontweight='bold', y=1.01)
plt.savefig(str(SAVE_DIR3 / 'all_results_overview.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Panel saved: all_results_overview.png')


## Step 11: Reward Distribution Histograms (1000 episodes)

In [ ]:
from src.agent import evaluate_trained_model_detailed

# get per-episode detail for final model
try:
    detail = evaluate_trained_model_detailed(env, final_model, n_episodes=1000)
    epi_rewards = detail['episode_rewards']
    epi_costs   = detail.get('episode_costs', [])
    epi_defects = detail.get('defect_escapes', [])
except Exception:
    # fallback: quick manual rollout
    epi_rewards, epi_costs = [], []
    obs, _  = env.reset()
    for _ in range(1000):
        step_rew = 0
        done = False
        while not done:
            action, _ = final_model.predict(obs, deterministic=True)
            obs, rew, done, trunc, info = env.step(int(action))
            step_rew += rew
            if done or trunc:
                epi_rewards.append(step_rew); epi_costs.append(info.get('cost_spent', 0))
                obs, _ = env.reset(); step_rew = 0; done = False; break

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(epi_rewards, bins=50, color='#3498DB', edgecolor='white', linewidth=0.5)
axes[0].axvline(float(np.mean(epi_rewards)), color='red', linewidth=2, label=f'Mean: {np.mean(epi_rewards):.2f}')
axes[0].axvline(best_bl, color='black', linewidth=2, linestyle='--', label=f'Baseline: {best_bl:.2f}')
axes[0].set_xlabel('Episode Reward'); axes[0].set_ylabel('Count')
axes[0].set_title('Final Model (1M steps) — Reward Distribution\n1000 episodes', fontsize=11, fontweight='bold')
axes[0].legend()

if epi_costs:
    axes[1].hist(epi_costs, bins=40, color='#2ECC71', edgecolor='white', linewidth=0.5)
    axes[1].axvline(float(np.mean(epi_costs)), color='red', linewidth=2, label=f'Mean: {np.mean(epi_costs):.1f}')
    axes[1].set_xlabel('Cost Spent per Episode'); axes[1].set_ylabel('Count')
    axes[1].set_title('Cost Budget Usage Distribution', fontsize=11, fontweight='bold')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Cost data unavailable', ha='center', va='center', transform=axes[1].transAxes)

# Stage comparison histograms overlay
bl_reward = [best_bl] * 100  # approximate
axes[2].hist(epi_rewards, bins=40, alpha=0.7, color='#3498DB', label=f'Final 1M (μ={np.mean(epi_rewards):.1f})')
for algo in top2_algos:
    seeds_r = [m['mean_reward'] for m in stage2_results[algo]]
    axes[2].axvline(np.mean(seeds_r), linewidth=2, linestyle='--', label=f'{algo.upper()} 500K (μ={np.mean(seeds_r):.1f})')
axes[2].axvline(best_bl, color='black', linewidth=2.5, linestyle=':', label=f'Baseline ({best_bl:.1f})')
axes[2].set_xlabel('Episode Reward'); axes[2].set_ylabel('Count')
axes[2].set_title('Comparison: Final vs Stage-2 vs Baseline', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(SAVE_DIR3 / 'reward_histograms.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: reward_histograms.png')


## Step 12: Algorithm Heatmap + Comprehensive Summary Table

In [ ]:
# Comprehensive metrics heatmap
stage1_df_full = pd.DataFrame(stage1_results).T.apply(pd.to_numeric, errors='coerce')
stage1_df_full['stage'] = 'Stage-1 (200K)'

stage2_df_list = []
for algo, seeds in stage2_results.items():
    for m in seeds:
        row = {k: float(v) if isinstance(v, (int, float)) else v for k, v in m.items()}
        row['algo'] = algo; row['stage'] = 'Stage-2 (500K)'
        stage2_df_list.append(row)
stage2_df_full = pd.DataFrame(stage2_df_list)

# Heatmap of stage1 metric matrix
metric_cols = [c for c in stage1_df_full.columns if c not in ['stage'] and stage1_df_full[c].dtype in [float, 'float64']]
if metric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    hm_data = stage1_df_full[metric_cols].apply(pd.to_numeric, errors='coerce').dropna(axis=1)
    if not hm_data.empty:
        normed = (hm_data - hm_data.min()) / (hm_data.max() - hm_data.min() + 1e-9)
        sns.heatmap(normed.T, annot=hm_data.T.round(3), fmt='g', cmap='YlGnBu',
                    ax=axes[0], cbar_kws={'label': 'Normalized [0-1)'}, linewidths=0.5)
        axes[0].set_title('Stage-1 Metric Heatmap (normalized)', fontsize=12, fontweight='bold')
    else:
        axes[0].text(0.5, 0.5, 'No numeric metrics in stage1', ha='center', va='center', transform=axes[0].transAxes)

    # Comprehensive summary table
    rows = []
    for name, r in baseline_results.items():
        rows.append({'Method': name, 'Stage': 'Baseline', 'Mean Reward': r.get('mean_reward',0),
                     'Std Reward': r.get('std_reward',0), 'Accuracy': r.get('accuracy',0), 'Steps': 0})
    for name, r in stage1_results.items():
        rows.append({'Method': name, 'Stage': 'Stage-1', 'Mean Reward': r.get('mean_reward',0),
                     'Std Reward': r.get('std_reward',0), 'Accuracy': r.get('accuracy',0), 'Steps': 200_000})
    for algo, seeds in stage2_results.items():
        rews = [m.get('mean_reward',0) for m in seeds]; accs = [m.get('accuracy',0) for m in seeds]
        rows.append({'Method': algo+'(3-seed)', 'Stage': 'Stage-2', 'Mean Reward': float(np.mean(rews)),
                     'Std Reward': float(np.std(rews)), 'Accuracy': float(np.mean(accs)), 'Steps': 500_000})
    rows.append({'Method': best_algo+'(HPO)', 'Stage': 'Final', 'Mean Reward': final_metrics.get('mean_reward',0),
                 'Std Reward': final_metrics.get('std_reward',0), 'Accuracy': final_metrics.get('accuracy',0), 'Steps': 1_000_000})
    summary_df = pd.DataFrame(rows)
    summary_df['Mean Reward'] = summary_df['Mean Reward'].round(2)
    summary_df['Std Reward']  = summary_df['Std Reward'].round(3)
    summary_df['Accuracy']    = summary_df['Accuracy'].round(3)
    print('\n=== COMPREHENSIVE SUMMARY TABLE ===' )
    print(summary_df.to_string(index=False))
    summary_df.to_csv(str(SAVE_DIR3 / 'comprehensive_summary.csv'), index=False)

    axes[1].axis('off')
    tbl = axes[1].table(cellText=summary_df.values, colLabels=summary_df.columns,
                        cellLoc='center', loc='center', bbox=[0,0,1,1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    axes[1].set_title('Comprehensive Summary Table', fontsize=12, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig(str(SAVE_DIR3 / 'heatmap_and_table.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: heatmap_and_table.png')
else:
    print('Skipping heatmap — no numeric metric columns found')


## Step 13: ZIP All Models + Export to /kaggle/working/

In [ ]:
import zipfile, hashlib, datetime

zip_path = SAVE_DIR3 / 'rl_test_flow_final_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ff in sorted(SAVE_DIR3.rglob('*')):
        if ff.is_file() and not ff.name.endswith('.zip'):
            zf.write(ff, ff.relative_to(SAVE_DIR3))

zip_sha = hashlib.md5(zip_path.read_bytes()).hexdigest()

print('=== NB3 FINAL ARTIFACTS ===')
total = 0
for ff in sorted(SAVE_DIR3.rglob('*')):
    if ff.is_file():
        print(f'  {str(ff.relative_to(SAVE_DIR3)):60s} {ff.stat().st_size/1e3:7.0f} KB')
        total += ff.stat().st_size
print(f'\nTotal output: {total/1e6:.1f} MB')
print(f'ZIP: {zip_path.name} | MD5: {zip_sha}')

# Print final metrics card
print('\n' + '='*60)
print('FINAL PERFORMANCE CARD')
print('='*60)
print(f'  Algorithm:         {best_algo.upper()} + Optuna HPO')
print(f'  Timesteps:         1,000,000')
print(f'  Mean Reward:       {final_metrics["mean_reward"]:+.2f}')
print(f'  Std Reward:        {final_metrics["std_reward"]:+.2f}')
print(f'  Accuracy:          {final_metrics["accuracy"]*100:.1f}%')
print(f'  Baseline (best):   {best_bl:+.2f}')
print(f'  Improvement:       +{final_metrics["mean_reward"]-best_bl:.2f}')
print(f'  Date:              {datetime.date.today()}')
print('='*60)
print('COMPLETE — Download output as Kaggle Dataset for future use')
